In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Helvetica"
})

import matplotlib as mpl

In [2]:
import toml
import warnings
import pandas as pd

def custom_formatwarning(msg, *args, **kwargs):
    # ignore everything except the message
    return str(msg) + '\n'

warnings.formatwarning = custom_formatwarning

## Tools for symmetry operations

__Desired properties:__

* Relating operator names to matrix operations
* Identification of orbits
* Constructing stabilizer groups

In [18]:
point_group_lib = toml.load('../../include/point_groups.toml')

In [19]:
symmetry_operators = toml.load('../../include/symmetry_operations.toml')

In [20]:
point_group_lib

{'C1': {'generators': ['E']},
 'Ci': {'generators': ['I']},
 'C2': {'generators': ['C2,001']},
 'Cs': {'generators': ['m,001']},
 'C2h': {'generators': ['C2,001', 'I']},
 'D2': {'generators': ['C2,010', 'C2,001']},
 'C2v': {'generators': ['C2,001', 'm,010']},
 'D2h': {'generators': ['C2,010', 'C2,001', 'I']},
 'C4': {'generators': ['C4p,001']},
 'S4': {'generators': ['S4p,001']},
 'C4h': {'generators': ['C4p,001', 'I']},
 'D4': {'generators': ['C4p,001', 'C2,010', 'C2,001']},
 'C4v': {'generators': ['C4p,001', 'C2,001', 'm,010']},
 'D2d': {'generators': ['S4p,001', 'C2,010', 'C2,001']},
 'D4h': {'generators': ['C4p,001', 'C2,010', 'C2,001', 'I']},
 'C3': {'generators': ['C3p,001']},
 'C3i': {'generators': ['S3p,001']},
 'D3': {'generators': ['C3p,001', 'C2,1-10']},
 'C3v': {'generators': ['C3p,001', 'm,110']},
 'D3d': {'generators': ['S6p,001', 'C2,1-10', 'I']},
 'C6': {'generators': ['C6p_H,001']},
 'C3h': {'generators': ['C3p_H,001', 'I']},
 'C6h': {'generators': ['C6p_H,001', 'I']},

In [98]:
def get_symmetry_matrix(element):
    """
    Translates the symmetry symbol, as appearing in symmetry_operators
    library, to a matrix operator in the lattice basis.

    Arguments:
    element - str, symmetry symbol written as '_operation_,_axis_'.

    Returns:
    matrix_operator - np.ndarray, matrix operator in the lattice basis.
    """
    operation, axis = element.split(',')

    # Check syntax
    axis_list = list(symmetry_operators.keys())

    if axis not in axis_list:
        raise NameError('Invalid axis: ' + axis + \
                        '\nSupported axes: ' + ", ".join(axis_list))

    operation_list = list(symmetry_operators[axis].keys())
    
    if operation not in operation_list:
        raise NameError('Invalid operation symbol: ' + operation + \
                        '\nSupported operations for the given symmetry axis '\
                        'are ' + ", ".join(operation_list))
    
    matrix_operator = np.array(symmetry_operators[axis][operation],int)

    return matrix_operator

In [115]:
def point_transform(point, generators):
    """
    Applies generators of the group to a single point. 
    Operations assume generators to be matrices of type np.2darray, and point
    of type np.1darray.

    Arguments:
    point            - np.1darray, 'seed' point for the transformation;
    generators       - array of np.2darray, generators of the group.

    Returns:
    orbit        - list, a set of partners generated from the seed point;
    transporters - list of np.2darray, a set of operators that transform the
                   seed to the other points in the orbit;
    stabilizers  - list of np.2darray, operators that leave the seed point
                   unchanged.
    """

    orbit = [point]
    transporters = [np.eye(len(generators[0]),dtype=int)]
    stabilizers = [np.eye(len(generators[0]),dtype=int)]

    counter = 0
    
    for p in orbit:
        for g in generators:
            new_point = g.dot(p)
            print(new_point)
            
            if not any((new_point==point).all() for point in orbit):
                orbit += [new_point]
                transporters += [transporters[counter].dot(g)]
                counter += 1

            else:
                
        
    return orbit, transporters, stabilizers

In [116]:
test_generator_symbols = point_group_lib['C4']['generators']
test_generators = [get_symmetry_matrix(test_generator_symbols[0])]

In [121]:
test_o, test_t, test_s = point_transform(np.array([0,0,1],int),c4_generators)

[0 0 1]


In [122]:
test_o

[array([0, 0, 1])]

In [123]:
test_t

[array([[1, 0, 0],
        [0, 1, 0],
        [0, 0, 1]])]

In [107]:
test_t[3].dot(test_o[0])

array([ 0., -1.,  0.])

In [86]:
test_s

[]